# Checkpoint 6 — subscriber-tier model A/B run

This notebook retrains the four independent XGBoost models with subscriber_tier one-hot encoded. Everything else is held constant: the same horizon rows, saved channel-grouped folds, target transform, outlier rule, model parameters, and untouched reserved test.

The new artifacts are saved separately so the tier-enabled result can be compared fairly with checkpoint 5.

In [1]:
from pathlib import Path
import json
import sys

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.train_checkpoint5_models import (
    HORIZONS,
    MODEL_NAME,
    load_horizon_checkpoint,
    train_all_horizons,
)
from viewcastlk_ml.horizon_preprocessing import (
    SUBSCRIBER_TIER_ORDER,
    HorizonDatasetPreprocessor,
)

OLD_ARTIFACT_DIR = PROJECT_ROOT / 'artifacts' / 'checkpoint5_xgboost'
NEW_ARTIFACT_DIR = PROJECT_ROOT / 'artifacts' / 'checkpoint6_subscriber_tier'

loaded = {}
tier_rows = []
for horizon in HORIZONS:
    X, y, assignments, _, _ = load_horizon_checkpoint(PROJECT_ROOT, horizon)
    loaded[horizon] = (X, y, assignments)
    assert 'subscriber_tier' in X.columns
    assert X['subscriber_tier'].notna().all()
    counts = X['subscriber_tier'].value_counts().reindex(SUBSCRIBER_TIER_ORDER, fill_value=0)
    for tier, count in counts.items():
        tier_rows.append({
            'horizon_days': horizon,
            'subscriber_tier': tier,
            'video_rows': int(count),
            'percent': 100 * count / len(X),
        })

tier_distribution = pd.DataFrame(tier_rows)
display(tier_distribution.pivot(
    index='subscriber_tier', columns='horizon_days', values='percent'
).reindex(SUBSCRIBER_TIER_ORDER).round(2))

horizon_days        7      14     21     30
subscriber_tier                            
under_1k          6.91   7.22   7.66   7.73
1k_to_10k        15.21  17.68  20.26  19.03
10k_to_100k      16.40  19.54  20.75  21.40
100k_to_1m       21.26  21.08  21.03  21.41
1m_plus          40.23  34.48  30.29  30.43
missing           0.00   0.00   0.00   0.00


In [2]:
# Prove that subscriber_tier is fitted only from fold training data and becomes numeric one-hot columns.
X7, y7, assignments7 = loaded[7]
train_mask = assignments7['partition'].eq('development') & ~assignments7['cv_validation_fold'].eq(1)
validation_mask = assignments7['partition'].eq('development') & assignments7['cv_validation_fold'].eq(1)
train_positions = assignments7.loc[train_mask, 'horizon_row_position'].astype(int).to_numpy()
validation_positions = assignments7.loc[validation_mask, 'horizon_row_position'].astype(int).to_numpy()

preprocessor = HorizonDatasetPreprocessor()
preprocessor.fit(X7.iloc[train_positions], np.log1p(y7.iloc[train_positions]))
preview_source = X7.iloc[validation_positions[:8]]
preview = preprocessor.transform(preview_source)
tier_feature_columns = [column for column in preview if column.startswith('subscriber_tier_')]

print('Tier model columns:', tier_feature_columns)
display(pd.concat([
    preview_source[['ch_subs_at_publish', 'subscriber_tier']].reset_index(drop=True),
    preview[tier_feature_columns].reset_index(drop=True),
], axis=1))

assert tier_feature_columns
assert np.allclose(preview[tier_feature_columns].sum(axis=1), 1.0)
assert 'subscriber_tier' not in preview.columns
print('PASS: every preview row has exactly one numeric subscriber-tier indicator.')

Tier model columns: ['subscriber_tier_100k_to_1m', 'subscriber_tier_10k_to_100k', 'subscriber_tier_1k_to_10k', 'subscriber_tier_1m_plus', 'subscriber_tier_under_1k']
   ch_subs_at_publish  ... subscriber_tier_under_1k
0           3770000.0  ...                      0.0
1           3770000.0  ...                      0.0
2           3780000.0  ...                      0.0
3              7340.0  ...                      0.0
4           3770000.0  ...                      0.0
5           3770000.0  ...                      0.0
6            589000.0  ...                      0.0
7           3780000.0  ...                      0.0

[8 rows x 7 columns]
PASS: every preview row has exactly one numeric subscriber-tier indicator.


In [3]:
training_run = train_all_horizons(
    project_root=PROJECT_ROOT,
    output_dir=NEW_ARTIFACT_DIR,
    n_estimators=800,
    n_jobs=4,
    include_llm_scores=False,
)

display(training_run['summary'][[
    'horizon_days', 'model', 'rows', 'mape_nonzero_pct',
    'median_ape_nonzero_pct', 'smape_pct', 'rmsle', 'log_r2'
]])


Training independent day-7 model
day 7 fold 1/5: RMSLE=1.8652, median APE=100.22%, best trees=102
day 7 fold 2/5: RMSLE=2.0680, median APE=85.84%, best trees=125
day 7 fold 3/5: RMSLE=2.0893, median APE=93.12%, best trees=150
day 7 fold 4/5: RMSLE=1.8518, median APE=85.18%, best trees=55
day 7 fold 5/5: RMSLE=2.2008, median APE=105.31%, best trees=40

Training independent day-14 model
day 14 fold 1/5: RMSLE=1.9231, median APE=132.30%, best trees=45
day 14 fold 2/5: RMSLE=2.1911, median APE=90.34%, best trees=1
day 14 fold 3/5: RMSLE=2.4621, median APE=98.53%, best trees=320
day 14 fold 4/5: RMSLE=1.9078, median APE=89.00%, best trees=62
day 14 fold 5/5: RMSLE=1.9972, median APE=93.78%, best trees=173

Training independent day-21 model
day 21 fold 1/5: RMSLE=1.9571, median APE=119.96%, best trees=29
day 21 fold 2/5: RMSLE=1.9133, median APE=84.93%, best trees=83
day 21 fold 3/5: RMSLE=1.9902, median APE=87.22%, best trees=44
day 21 fold 4/5: RMSLE=1.9765, median APE=90.14%, best trees=

In [4]:
# A/B comparison: positive change means the subscriber-tier model improved.
old_summary = pd.read_csv(OLD_ARTIFACT_DIR / 'cv_summary_metrics.csv', dtype={'horizon_days': str})
new_summary = pd.read_csv(NEW_ARTIFACT_DIR / 'cv_summary_metrics.csv', dtype={'horizon_days': str})
comparison_rows = []
for horizon in [str(h) for h in HORIZONS] + ['combined']:
    old = old_summary[
        old_summary['horizon_days'].eq(horizon)
        & old_summary['model'].eq(MODEL_NAME)
    ].iloc[0]
    new = new_summary[
        new_summary['horizon_days'].eq(horizon)
        & new_summary['model'].eq(MODEL_NAME)
    ].iloc[0]
    comparison_rows.append({
        'horizon_days': horizon,
        'old_rmsle': old['rmsle'],
        'tier_rmsle': new['rmsle'],
        'rmsle_improvement_pct': 100 * (old['rmsle'] - new['rmsle']) / old['rmsle'],
        'old_mape_pct': old['mape_nonzero_pct'],
        'tier_mape_pct': new['mape_nonzero_pct'],
        'mape_improvement_pct': 100 * (old['mape_nonzero_pct'] - new['mape_nonzero_pct']) / old['mape_nonzero_pct'],
        'old_log_r2': old['log_r2'],
        'tier_log_r2': new['log_r2'],
    })

comparison = pd.DataFrame(comparison_rows)
display(comparison.round(4))

  horizon_days  old_rmsle  ...  old_log_r2  tier_log_r2
0            7     2.0716  ...      0.3005       0.3352
1           14     2.1144  ...      0.2991       0.3043
2           21     2.0166  ...      0.3665       0.3564
3           30     2.0487  ...      0.3753       0.3825
4     combined     2.0644  ...      0.3329       0.3439

[5 rows x 9 columns]


In [5]:
# Artifact and leakage tests. No reserved test prediction is made.
manifest = json.loads((NEW_ARTIFACT_DIR / 'training_manifest.json').read_text(encoding='utf-8'))
predictions = pd.read_csv(NEW_ARTIFACT_DIR / 'cv_predictions.csv')
old_predictions = pd.read_csv(OLD_ARTIFACT_DIR / 'cv_predictions.csv')
test_rows = []

def check(name, condition, detail=''):
    test_rows.append({'test': name, 'status': 'PASS' if bool(condition) else 'FAIL', 'detail': detail})

check('reserved test remains unevaluated', manifest['status'] == 'candidate_reserved_test_not_evaluated')
check('all predictions finite', np.isfinite(predictions.filter(like='predicted_').to_numpy(dtype=float)).all())
check('A/B uses identical OOF rows', set(zip(predictions['horizon_days'], predictions['horizon_row_position'])) == set(zip(old_predictions['horizon_days'], old_predictions['horizon_row_position'])))

for record in manifest['models']:
    horizon = record['horizon_days']
    bundle = joblib.load(NEW_ARTIFACT_DIR / record['model_path'])
    tier_columns = [feature for feature in bundle.feature_names if feature.startswith('subscriber_tier_')]
    X, y, assignments = loaded[horizon]
    development_positions = set(assignments.loc[assignments['partition'].eq('development'), 'horizon_row_position'].astype(int))
    reserved_positions = set(assignments.loc[assignments['partition'].eq('test_reserved'), 'horizon_row_position'].astype(int))
    predicted_positions = set(predictions.loc[predictions['horizon_days'].eq(horizon), 'horizon_row_position'].astype(int))
    sample_position = min(development_positions)
    sample_prediction = bundle.predict_views(X.iloc[[sample_position]])

    check(f'day {horizon} tier indicators saved in model', len(tier_columns) >= 5, ', '.join(tier_columns))
    check(f'day {horizon} development coverage exact', predicted_positions == development_positions)
    check(f'day {horizon} reserved rows absent', predicted_positions.isdisjoint(reserved_positions))
    check(f'day {horizon} bundle reloads and predicts', len(sample_prediction) == 1 and np.isfinite(sample_prediction).all())

tests = pd.DataFrame(test_rows)
display(tests)
failures = tests[tests['status'].eq('FAIL')]
assert failures.empty, failures.to_string(index=False)
print(f'PASS: all {len(tests)} subscriber-tier model checks succeeded.')

                                     test  ...                                             detail
0       reserved test remains unevaluated  ...                                                   
1                  all predictions finite  ...                                                   
2             A/B uses identical OOF rows  ...                                                   
3    day 7 tier indicators saved in model  ...  subscriber_tier_100k_to_1m, subscriber_tier_10...
4        day 7 development coverage exact  ...                                                   
5              day 7 reserved rows absent  ...                                                   
6       day 7 bundle reloads and predicts  ...                                                   
7   day 14 tier indicators saved in model  ...  subscriber_tier_100k_to_1m, subscriber_tier_10...
8       day 14 development coverage exact  ...                                                   
9             day 14

## Checkpoint decision

Use the A/B table to decide whether subscriber_tier should remain a model feature. The raw subscriber count remains available either way. This run does not yet introduce tier-balanced splitting or channel-frequency sample weighting.